# 03 - Abramson et al. 2024: AlphaFold3 Reproduction

This notebook is a **from-scratch PyTorch/RDKit reproduction** of AlphaFold3's central ideas. It does not use the official AF3 code, API, Docker image, or model parameters.

We assume RDKit is installed, as requested. RDKit is used for ligand parsing, atom/bond features, conformers, stereochemistry, and chemistry-aware benchmark preparation.

AF3's conceptual steps:

1. Represent all biomolecules as tokens and atoms: proteins, nucleic acids, ligands, ions, and modified residues.
2. Build pair and single representations with molecule-aware features.
3. Use Pairformer-style pair/single updates.
4. Predict coordinates with a diffusion-style denoising model.
5. Score across protein, nucleic-acid, ligand, and interface metrics.

## Layout and assumptions

Your A100 80GB matches the class of GPU needed for serious AF3-style inference experiments. The notebook assumes RDKit will be installed before execution on the cluster. If the import fails, install RDKit first and rerun; this notebook intentionally does not fall back to a non-chemistry path.

In [ ]:
from __future__ import annotations

# Shared notebook setup: all three reproductions use the same project layout,
# deterministic seeds, and cluster-aware device selection. This is not a paper
# method; it is experiment hygiene so Senior/Jumper/Abramson runs are comparable.
import json
import math
import os
import random
import shlex
import subprocess
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
MODEL_ROOT = PROJECT_ROOT / "models"
RESULTS_ROOT = PROJECT_ROOT / "results"
RUNS_ROOT = PROJECT_ROOT / "runs"

# Create the four persistent experiment roots used throughout the notebooks.
for path in [DATA_ROOT, MODEL_ROOT, RESULTS_ROOT, RUNS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

def latest_environment_report() -> dict:
    """Load the newest hardware report produced by Environment_Hardware_Check.ipynb."""
    report_dir = DATA_ROOT / "environment_reports"
    reports = sorted(report_dir.glob("environment_report_*_utc.json"))
    if not reports:
        return {}
    return json.loads(reports[-1].read_text(encoding="utf-8"))

ENV_REPORT = latest_environment_report()

def seed_everything(seed: int = 7) -> None:
    """Set Python/NumPy/PyTorch seeds so smoke runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def device() -> torch.device:
    """Prefer the first CUDA GPU on the cluster, otherwise fall back to CPU."""
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def write_text(path: Path, text: str) -> Path:
    """Write a UTF-8 text artifact after creating its parent directory."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")
    return path

def run(cmd: list[str], *, cwd: Path | None = None, timeout: int = 30, dry_run: bool = True):
    """Small command runner used for optional external tools and SLURM helpers."""
    print("$", " ".join(shlex.quote(str(x)) for x in cmd))
    if dry_run:
        print("DRY_RUN=True, command was not executed.")
        return None
    completed = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout, check=False)
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")
    return completed

def gpu_summary() -> str:
    """Summarize the saved GPU report so notebook output records the hardware."""
    devices = ENV_REPORT.get("torch", {}).get("devices", [])
    if devices:
        first = devices[0]
        return f"{first.get('name')} / {first.get('total_memory_gb')} GB / cc {first.get('major')}.{first.get('minor')}"
    return "No saved GPU report found."

seed_everything(7)
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device()}")
print(f"Saved cluster GPU: {gpu_summary()}")

In [ ]:
PAPER = "abramson_2024"
DATA_DIR = DATA_ROOT / PAPER
MODEL_DIR = MODEL_ROOT / PAPER
RESULT_DIR = RESULTS_ROOT / PAPER
RUN_DIR = RUNS_ROOT / PAPER

# Abramson et al. / AlphaFold3 expands the data surface beyond proteins:
# ligands, complexes, benchmarks, predictions, and metrics each need explicit
# storage because the paper evaluates multimolecular interactions.
paths = {
    "raw": DATA_DIR / "raw",
    "features": DATA_DIR / "features",
    "ligands": DATA_DIR / "ligands",
    "benchmarks": DATA_DIR / "benchmarks",
    "checkpoints": MODEL_DIR / "checkpoints",
    "predictions": RESULT_DIR / "predictions",
    "metrics": RESULT_DIR / "metrics",
    "slurm": RUN_DIR / "slurm",
}
for p in paths.values():
    p.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in paths.items()}, indent=2))

## Step 1 - RDKit chemistry layer

AF3's protein-ligand benchmark surface is impossible to treat seriously without chemistry. RDKit gives us explicit atoms, formal charges, aromaticity, hybridization, bonds, stereochemistry, and conformers. Those features become ligand tokens/atoms for our own model.

Scientifically, ligands are not just residue-like tokens: bond order, charge, aromaticity, stereochemistry, and protonation state determine which poses are chemically possible. A model that ignores those constraints can appear close by RMSD while predicting an invalid molecule.

Computationally, RDKit turns SMILES/SDF chemistry into graph features and initial conformers. Mathematically, the ligand is represented as a graph `G = (V, E)` with atom feature matrix `X_v`, bond feature matrix `X_e`, and coordinates `R`. These features become conditioning variables for the neural coordinate model.

In [ ]:
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem

print("RDKit version:", rdkit.__version__)

# Atom/bond vocabularies for the AF3-style ligand featurization layer. In
# Abramson et al., small molecules are represented with chemistry-aware atom
# and bond information rather than residue-only sequence tokens.
ATOM_TYPES = ["C", "N", "O", "S", "P", "F", "Cl", "Br", "I", "B", "H", "OTHER"]
BOND_TYPES = [
    Chem.BondType.SINGLE,
    Chem.BondType.DOUBLE,
    Chem.BondType.TRIPLE,
    Chem.BondType.AROMATIC,
]

def one_hot_index(value, choices):
    """Encode categorical chemistry values, using the final bin as fallback."""
    idx = choices.index(value) if value in choices else len(choices) - 1
    out = torch.zeros(len(choices))
    out[idx] = 1.0
    return out

def featurize_ligand_smiles(smiles: str, seed: int = 7):
    """Convert SMILES into AF3-style ligand atom, bond, and conformer features.

    This reproduces the AF3 paper's need for chemistry-aware ligand inputs using
    RDKit primitives: sanitization/parsing, hydrogens, bonds, aromaticity,
    formal charge, and an initial 3D conformer.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Could not parse SMILES: {smiles}")
    mol = Chem.AddHs(mol)
    # RDKit conformer generation supplies initial ligand geometry for the
    # notebook's toy complex; real AF3 reproduction should use curated SDF/mmCIF.
    status = AllChem.EmbedMolecule(mol, randomSeed=seed)
    if status == 0:
        AllChem.UFFOptimizeMolecule(mol, maxIters=200)
    conf = mol.GetConformer()
    atom_features, coords = [], []
    for atom in mol.GetAtoms():
        # Atom features encode the chemical constraints that govern valid poses.
        symbol = atom.GetSymbol()
        atom_features.append(torch.cat([
            one_hot_index(symbol, ATOM_TYPES),
            torch.tensor([
                atom.GetFormalCharge(),
                float(atom.GetIsAromatic()),
                atom.GetTotalDegree(),
                atom.GetTotalValence(),
            ], dtype=torch.float32),
        ]))
        p = conf.GetAtomPosition(atom.GetIdx())
        coords.append([p.x, p.y, p.z])
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        # Store both bond directions so later graph/message-passing code can
        # consume ligand connectivity without assuming undirected edges.
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        feat = one_hot_index(bond.GetBondType(), BOND_TYPES)
        edge_index += [[i, j], [j, i]]
        edge_attr += [feat, feat]
    return {
        "atom_features": torch.stack(atom_features),
        "atom_coords": torch.tensor(coords, dtype=torch.float32),
        "edge_index": torch.tensor(edge_index, dtype=torch.long).T if edge_index else torch.empty(2, 0, dtype=torch.long),
        "edge_attr": torch.stack(edge_attr) if edge_attr else torch.empty(0, len(BOND_TYPES)),
        "canonical_smiles": Chem.MolToSmiles(Chem.RemoveHs(mol)),
    }

lig = featurize_ligand_smiles("CC(=O)OC1=CC=CC=C1C(=O)O")
print({k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in lig.items()})

## Step 2 - Multimolecular tokenization

AF3 treats proteins, nucleic acids, and ligands in one coordinate-generating system. This simplified tokenizer creates:

- Protein residue tokens from amino-acid sequence.
- Ligand atom tokens from RDKit atom features.
- Pair features from sequence distance, molecule identity, and ligand bonds.

This is enough to train a tiny denoising model now and swap in real benchmark complexes later.

Scientifically, AF3's key expansion is that proteins, nucleic acids, ligands, ions, and modifications share one interaction model. Tokenization is where we decide what the model is allowed to know about each molecule and how cross-molecule contacts can be represented.

Computationally, this step concatenates protein residue tokens and ligand atom tokens, then builds pair features for sequence distance, molecule identity, and ligand bonds. Mathematically, we create a single token set of size `T` with single features `S in R^{T x C}` and pair features `P in R^{T x T x C_p}`; the model's job is to infer coordinates for all tokens jointly.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWYX"

def tokenize_complex(sequence: str, ligand_smiles: str):
    """Create AF3-like single/pair tensors for a protein-ligand complex.

    Abramson et al. use a unified token system for proteins, nucleic acids,
    ligands, ions, and modifications. This simplified tokenizer concatenates
    protein residue tokens with RDKit ligand atom tokens and adds pair features.
    """
    protein_tokens = F.one_hot(torch.tensor([AA.index(a) if a in AA else AA.index("X") for a in sequence]), len(AA)).float()
    lig = featurize_ligand_smiles(ligand_smiles)
    # Pad/truncate ligand atom features so protein residues and ligand atoms can
    # share one single-token tensor.
    ligand_tokens = F.pad(lig["atom_features"], (0, protein_tokens.shape[-1] - lig["atom_features"].shape[-1])) if lig["atom_features"].shape[-1] < protein_tokens.shape[-1] else lig["atom_features"][:, :protein_tokens.shape[-1]]
    single = torch.cat([protein_tokens, ligand_tokens], dim=0)
    molecule_id = torch.cat([torch.zeros(len(protein_tokens), dtype=torch.long), torch.ones(len(ligand_tokens), dtype=torch.long)])
    L = single.shape[0]
    pair = torch.zeros(L, L, 8)
    idx = torch.arange(L)
    # Pair channels encode sequence separation, same/different molecule, and
    # ligand bond adjacency: a small analogue of AF3's rich pair representation.
    pair[..., 0] = (idx[:, None] - idx[None, :]).abs().float().clamp(max=32) / 32
    pair[..., 1] = (molecule_id[:, None] == molecule_id[None, :]).float()
    pair[..., 2] = (molecule_id[:, None] != molecule_id[None, :]).float()
    offset = len(protein_tokens)
    for e in lig["edge_index"].T:
        # Ligand bond connectivity is written into pair features after offsetting
        # ligand atom indices behind the protein residue tokens.
        i, j = int(e[0]) + offset, int(e[1]) + offset
        pair[i, j, 3] = 1.0
    true_coords = torch.cat([
        # Toy coordinates: protein as a noisy CA trace and ligand conformer offset
        # nearby. Real AF3 reproduction should use experimental complex geometry.
        torch.cumsum(torch.randn(len(protein_tokens), 3) * 0.4 + torch.tensor([3.8, 0.0, 0.0]), dim=0),
        lig["atom_coords"] + torch.tensor([0.0, 8.0, 0.0]),
    ], dim=0)
    return {"single": single, "pair": pair, "true_coords": true_coords, "molecule_id": molecule_id}

complex_features = tokenize_complex("MSTNPKPQRKTKRNTNRRPQDVKFPGG", "CCO")
print({k: tuple(v.shape) for k, v in complex_features.items()})

## Step 3 - Pairformer-style model

AF3 simplifies AF2's Evoformer into Pairformer-style updates over single and pair representations. This compact version alternates single self-attention with pair-conditioned updates. It is intentionally small but structurally honest: the model reasons over cross-molecule pair features before denoising atom coordinates.

Scientifically, interactions are pairwise and contextual: a ligand atom's placement depends on nearby protein residues, and a residue's relevance depends on the ligand pose. Pairformer-style updates give the network a place to represent these cross-molecule hypotheses.

Computationally, single-token attention mixes information across the complex, while pair updates maintain explicit `T x T` relational state. Mathematically, the block alternates updates to `S_i` and `P_ij`; the diffusion head then predicts coordinate noise from a combination of token state and averaged pair context.

In [ ]:
class PairformerLiteBlock(nn.Module):
    """Compact Pairformer-style update block inspired by Abramson et al. AF3.

    AF3 replaces AF2's Evoformer with Pairformer-style single/pair processing
    for multimolecular systems. This block keeps single-token attention and
    explicit pair updates, but omits many production details.
    """

    def __init__(self, single_dim: int, pair_dim: int, heads: int = 4):
        """Create single-token attention and pair-conditioned update layers."""
        super().__init__()
        # Single attention lets protein residues and ligand atoms exchange
        # context across the whole complex.
        self.single_attn = nn.MultiheadAttention(single_dim, heads, batch_first=True)
        self.pair_to_bias = nn.Linear(pair_dim, heads)
        self.single_ff = nn.Sequential(nn.LayerNorm(single_dim), nn.Linear(single_dim, 4 * single_dim), nn.GELU(), nn.Linear(4 * single_dim, single_dim))
        self.pair_ff = nn.Sequential(nn.LayerNorm(pair_dim), nn.Linear(pair_dim + single_dim, 4 * pair_dim), nn.GELU(), nn.Linear(4 * pair_dim, pair_dim))

    def forward(self, single, pair):
        """Update single and pair representations for one Pairformer-lite layer."""
        attn, _ = self.single_attn(single, single, single, need_weights=False)
        single = single + attn
        single = single + self.single_ff(single)
        pair_single = single[:, :, None, :] + single[:, None, :, :]
        # Pair update conditions relational state on the current token states.
        pair = pair + self.pair_ff(torch.cat([pair, pair_single], dim=-1))
        return single, pair


class AF3DiffusionLite(nn.Module):
    """AF3-like Pairformer plus coordinate-denoising head.

    Abramson et al. use diffusion-style coordinate generation for complexes.
    This model predicts coordinate noise from noisy coordinates plus single/pair
    biomolecular features.
    """

    def __init__(self, token_in: int = 21, pair_in: int = 8, single_dim: int = 128, pair_dim: int = 128, blocks: int = 6):
        """Create AF3-lite embeddings, Pairformer trunk, and denoising heads."""
        super().__init__()
        self.single_embed = nn.Linear(token_in, single_dim)
        self.pair_embed = nn.Linear(pair_in, pair_dim)
        # Time embedding conditions the denoiser on the current diffusion noise level.
        self.time_embed = nn.Sequential(nn.Linear(1, single_dim), nn.SiLU(), nn.Linear(single_dim, single_dim))
        self.blocks = nn.ModuleList([PairformerLiteBlock(single_dim, pair_dim) for _ in range(blocks)])
        self.noise_head = nn.Sequential(nn.LayerNorm(single_dim + pair_dim), nn.Linear(single_dim + pair_dim, 256), nn.SiLU(), nn.Linear(256, 3))
        # Confidence head placeholder mirrors AF3's confidence/scoring outputs.
        self.confidence_head = nn.Sequential(nn.LayerNorm(single_dim), nn.Linear(single_dim, 50))

    def forward(self, single, pair, noisy_coords, t):
        """Predict denoising vectors for all complex tokens at noise level `t`."""
        single = self.single_embed(single) + self.time_embed(t[:, None, None].float())
        pair = self.pair_embed(pair)
        # Current noisy inter-token distances are fed into the pair state, giving
        # the denoiser access to the geometry it is refining.
        coord_dist = torch.cdist(noisy_coords, noisy_coords)[..., None] / 20.0
        pair = pair + F.pad(coord_dist, (0, pair.shape[-1] - 1))
        for block in self.blocks:
            single, pair = block(single, pair)
        pair_context = pair.mean(dim=2)
        noise = self.noise_head(torch.cat([single, pair_context], dim=-1))
        confidence = self.confidence_head(single)
        return {"pred_noise": noise, "confidence_logits": confidence, "single": single, "pair": pair}

model = AF3DiffusionLite().to(device())
item = {k: v[None].to(device()) for k, v in complex_features.items() if k in ["single", "pair", "true_coords"]}
# Shape check for AF3-lite denoising outputs on a toy protein-ligand complex.
with torch.no_grad():
    t = torch.tensor([0.5], device=device())
    noisy = item["true_coords"] + torch.randn_like(item["true_coords"]) * t[:, None, None]
    out = model(item["single"], item["pair"], noisy, t)
print({k: tuple(v.shape) for k, v in out.items() if torch.is_tensor(v)})

## Step 4 - Diffusion training objective

The diffusion module learns to remove noise from atom coordinates conditioned on molecule features. We sample a noise level `t`, corrupt the true coordinates, and train the model to predict the injected noise. Later iterations should add AF3-style multi-step sampling, confidence calibration, bond/angle violation losses, clash losses, and molecule-class-specific metrics.

Scientifically, diffusion is useful because biomolecular complexes can have multiple plausible poses and conformations. Instead of predicting one deterministic coordinate set, the model learns a denoising process that can sample structures from a learned distribution.

Computationally, each training example draws random noise and a noise scale, which creates many supervised denoising tasks from the same structure. Mathematically, we sample `epsilon ~ N(0, I)`, form `x_t = x_0 + t epsilon`, and minimize `||epsilon_hat_theta(x_t, t, features) - epsilon||^2`. This is the score-matching core of the simplified diffusion objective.

In [ ]:
class AF3ToyComplexDataset(Dataset):
    """Tiny RDKit-backed protein-ligand dataset for AF3 diffusion smoke training."""

    def __init__(self, n: int = 8):
        """Build a small list of toy protein-ligand examples."""
        # Toy complexes cover several ligand graph shapes so RDKit featurization,
        # tokenization, and diffusion batches exercise different code paths.
        self.rows = [
            ("MSTNPKPQRKTKRNTNRRPQDVKFPGG", "CCO"),
            ("ACDEFGHIKLMNPQRSTVWY", "CC(=O)O"),
            ("GGGGSGGGGSGGGGS", "c1ccccc1"),
            ("MEEPQSDPSVEPPLSQETFSDLWKLL", "CCN(CC)CC"),
        ] * ((n + 3) // 4)
        self.rows = self.rows[:n]

    def __len__(self):
        """Return number of toy complexes."""
        return len(self.rows)

    def __getitem__(self, idx):
        """Tokenize one protein-ligand row into AF3-like tensors."""
        sequence, smiles = self.rows[idx]
        return tokenize_complex(sequence, smiles)

def collate_one(batch):
    """Batch-size-one collator until variable-length padding/masking is added."""
    # Keep batch size 1 until padding/masking is added.
    return {k: v[None] for k, v in batch[0].items()}

toy = AF3ToyComplexDataset()
loader = DataLoader(toy, batch_size=1, shuffle=True, collate_fn=collate_one)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
model.train()
# Diffusion training loop: sample noise level, corrupt coordinates, and train the
# model to predict the injected noise, matching the AF3-style denoising objective.
for step, item in enumerate(loader):
    item = {k: v.to(device()) for k, v in item.items()}
    t = torch.rand(item["true_coords"].shape[0], device=device()).clamp_min(0.05)
    noise = torch.randn_like(item["true_coords"])
    # Forward noising process: x_t = x_0 + t * epsilon.
    noisy = item["true_coords"] + noise * t[:, None, None]
    opt.zero_grad(set_to_none=True)
    outputs = model(item["single"], item["pair"], noisy, t)
    denoise_loss = F.mse_loss(outputs["pred_noise"], noise)
    opt.zero_grad(set_to_none=True)
    denoise_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    print(f"step={step} denoise_loss={float(denoise_loss.detach().cpu()):.4f}")
    if step >= 2:
        break

## Step 5 - Sampling

A trained diffusion model starts from noisy coordinates and repeatedly denoises. This minimal sampler is Euler-style and intentionally simple. The important point is that generation is ours: PyTorch tensors in, PyTorch tensors out, with RDKit-derived ligand features.

Scientifically, sampling turns the learned distribution into concrete candidate complexes. Multiple samples are important for AF3-like tasks because ligand binding poses, flexible loops, and interfaces may be ambiguous from sequence and chemistry alone.

Computationally, the sampler runs the neural network several times while reducing noise. Mathematically, it approximates the reverse denoising trajectory with discrete updates `x_{t-dt} = x_t - dt * epsilon_hat_theta(...)`; later we can replace this with a better scheduler and sample reranking.

In [ ]:
@torch.no_grad()
def sample_complex(model, features, steps: int = 20):
    """Generate complex coordinates by iterative AF3-lite denoising."""
    model.eval()
    single = features["single"][None].to(device())
    pair = features["pair"][None].to(device())
    coords = torch.randn_like(features["true_coords"][None].to(device()))
    for s in reversed(range(steps)):
        # Euler-style reverse diffusion step. This is intentionally simple; later
        # versions should add better schedulers, multiple samples, and reranking.
        t = torch.full((1,), (s + 1) / steps, device=device())
        pred_noise = model(single, pair, coords, t)["pred_noise"]
        coords = coords - pred_noise / steps
    return coords[0].detach().cpu()

sampled = sample_complex(model, complex_features, steps=5)
print(tuple(sampled.shape), sampled[:3])

## Step 6 - AF3 benchmark surfaces

For AF3, protein TM-score alone is insufficient. The score table must cover protein, ligand, nucleic acid, and interface quality. RDKit will also support chemistry validity checks for ligand predictions.

Scientifically, AF3's claim is about biomolecular interactions, so each molecular class needs metrics aligned with its biology. A protein backbone can be globally correct while the ligand pose is wrong, or a ligand can be close while the interface chemistry is invalid.

Computationally, we store a benchmark manifest that maps each target to required input files and metric families. Mathematically, the final evaluation is a vector-valued score, not a scalar: global fold terms, local/interface terms, ligand RMSD, clash/validity terms, and calibration terms should be tracked separately before any aggregate is reported.

In [ ]:
# Benchmark manifest records the multimolecular score surface required for AF3:
# protein fold quality, ligand pose, interface quality, clashes, and chemistry.
benchmark_manifest = [
    {
        "target_id": "protein_ligand_example",
        "inputs": {"protein_fasta": "data/abramson_2024/raw/protein_ligand_example.fasta", "ligand_sdf": "data/abramson_2024/ligands/protein_ligand_example.sdf"},
        "truth_cif": "data/abramson_2024/benchmarks/protein_ligand_example.cif",
        "metrics": ["protein_tm_score", "ligand_rmsd", "interface_lddt", "clash_count", "rdkit_sanitizes"],
        "faithful": True,
    },
    {
        "target_id": "protein_nucleic_acid_example",
        "inputs": {"complex_fasta": "data/abramson_2024/raw/protein_nucleic_acid_example.fasta"},
        "truth_cif": "data/abramson_2024/benchmarks/protein_nucleic_acid_example.cif",
        "metrics": ["interface_lddt", "nucleic_acid_rmsd", "clash_count"],
        "faithful": True,
    },
]
write_text(paths["benchmarks"] / "benchmark_manifest.example.json", json.dumps(benchmark_manifest, indent=2))
print(json.dumps(benchmark_manifest, indent=2))

## Step 7 - Improvement track

The baseline is our own Pairformer/diffusion implementation. The first improvement arms should focus on places where AF3's benchmark is sensitive:

1. More diffusion samples and better confidence reranking.
2. RDKit ligand state ensembles: protonation, tautomer, stereochemistry, conformers.
3. Cross-molecule pair features for contacts, bonds, covalent links, and templates.
4. Chemistry validity losses and post-sampling restrained minimization.
5. Class-specific scoring so ligand improvements do not hide protein regressions.

Scientifically, the improvement loop tests hypotheses about what limits performance: sampling diversity, ligand chemistry states, cross-molecule features, or physical validity. Each arm should correspond to a biological or modeling reason, not just a parameter tweak.

Computationally, each experiment is a reproducible transformation of data, model, sampler, or scorer. Mathematically, we compare metric vectors per target and per molecule class, looking for consistent positive deltas rather than a single averaged number that could hide regressions.

In [ ]:
# Experiment registry separates faithful AF3-lite reproduction attempts from
# enhanced sampling, ligand-state, and chemistry-loss experiments.
experiments = [
    {"name": "af3_lite_faithful_single_sample", "faithful": True, "change": "own Pairformer/diffusion baseline"},
    {"name": "af3_lite_8_sample_rerank", "faithful": False, "change": "more diffusion samples plus confidence/validity reranking"},
    {"name": "af3_lite_ligand_state_ensemble", "faithful": False, "change": "RDKit tautomer/protonation/conformer ensemble"},
    {"name": "af3_lite_chemistry_losses", "faithful": False, "change": "bond, clash, chirality, and validity losses"},
]
write_text(RUN_DIR / "experiment_registry.json", json.dumps(experiments, indent=2))
print(json.dumps(experiments, indent=2))

## Cluster execution template

The notebooks are deliberately importable and runnable from `papermill`, `jupyter nbconvert --execute`, or ordinary notebook execution. For long runs on the cluster, the code writes a small SLURM script that executes this notebook with parameters rather than keeping GPU time tied to the browser session.

In [ ]:
# SLURM wrapper for longer AF3-lite notebook executions on the GPU cluster.
slurm = paths["slurm"] / "train_af3_lite.sbatch"
slurm.write_text(f"""#!/usr/bin/env bash
#SBATCH --job-name=af3-lite
#SBATCH --partition=gpu
#SBATCH --gres=gpu:1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=48:00:00
#SBATCH --output={paths['slurm']}/%x-%j.out
set -euo pipefail
cd "{PROJECT_ROOT}"
jupyter nbconvert --to notebook --execute 03_Abramson_2024_AlphaFold3_Interaction_Reproduction.ipynb --output results/abramson_2024/executed.ipynb
""", encoding="utf-8")
print(slurm.read_text(encoding="utf-8"))